# 📊 Teaching the Translator: Training, Metrics & System Design

**Goal:** Implement BLEU, ROUGE, and METEOR from scratch, understand how translation models are trained (MLM + supervised fine-tuning), and design the full production system.

**Vibe:** You're a teacher grading your student translator. First they studied general language (pretraining), then learned from example translations (fine-tuning). Now you need to grade their work — and figure out how to deploy the whole operation. 📝✍️🚀

---

## 🎭 Masked Language Modeling: The Fill-in-the-Blank Game

### How Do You Teach a Model Language Without Any Labels?

Imagine you're learning a new language. Your teacher hands you books with some words blacked out:

```
"The _____ sat on the mat and licked its paws."

What goes in the blank? Options: cat, dog, rock, airplane...
```

You don't need anyone to TELL you the answer. You can figure it out from context! "sat on the mat", "licked its paws" — that's clearly a cat (or maybe a dog).

**This is Masked Language Modeling (MLM)** — and it's how BERT-style encoder models pretrain.

### The MLM Recipe 🍳

```
STEP 1: Take a sentence from the internet
   "The weather in Paris is beautiful in springtime"

STEP 2: Randomly mask 15% of tokens
   "The weather in [MASK] is beautiful in [MASK]"

STEP 3: Ask the model to predict the masked tokens
   Model predicts: [MASK_1] = "Paris", [MASK_2] = "springtime"

STEP 4: Loss = cross-entropy between prediction and actual masked tokens
   Correct! Loss goes down. Model learned something. 🎉

REPEAT this on billions of sentences...
```

### The 15% Rule (BERT's Specific Strategy)

BERT doesn't just blindly mask everything. For each chosen token:

| Strategy | Probability | Why |
|---|---|---|
| Replace with `[MASK]` | 80% | Main training signal — predict the word |
| Replace with random word | 10% | Forces model to check context, not just fill blanks |
| Keep the original | 10% | Model must learn to represent real tokens too |

### Why MLM Instead of Next-Token Prediction?

```
NEXT-TOKEN (GPT-style):  "The cat sat on the [PREDICT: mat]"
  → The model only looks LEFT. Unidirectional.
  → Fine for generation, bad for understanding.

MASKED LM (BERT-style):  "The [MASK] sat on the mat"
  → The model looks BOTH directions! "the ___ sat" + "___ on the mat"
  → Bidirectional context → better understanding.
  → Perfect for the encoder in translation!
```

> 🎯 **Staff insight:** For the encoder in an encoder-decoder translation model, we WANT bidirectional pretraining (MLM). Models like mBART pretrain both encoder and decoder using a denoising objective — corrupting text in various ways and training the encoder-decoder pair to reconstruct it. This gives a warm start for translation fine-tuning.


In [ ]:
"""
🎭 MLM from Scratch: The Fill-in-the-Blank Game

We'll implement a simple MLM data pipeline:
1. Tokenize sentences
2. Randomly mask 15% of tokens (80/10/10 split)
3. Show what the model sees vs what it needs to predict
"""

import random
import numpy as np
from collections import Counter

random.seed(42)
np.random.seed(42)


def build_simple_vocab(sentences):
    """Build a word-level vocabulary from a list of sentences."""
    special = ['[PAD]', '[UNK]', '[MASK]', '[CLS]', '[SEP]']
    word_counts = Counter()
    for sentence in sentences:
        word_counts.update(sentence.lower().split())
    vocab = special + [word for word, _ in word_counts.most_common()]
    token2id = {tok: i for i, tok in enumerate(vocab)}
    id2token = {i: tok for tok, i in token2id.items()}
    return token2id, id2token


def tokenize(sentence, token2id):
    """Tokenize a sentence (word-level for simplicity)."""
    tokens = ['[CLS]'] + sentence.lower().split() + ['[SEP]']
    ids = [token2id.get(t, token2id['[UNK]']) for t in tokens]
    return tokens, ids


def apply_mlm_masking(token_ids, token2id, id2token, mask_prob=0.15):
    """
    Apply BERT-style MLM masking:
    - 15% of tokens are selected for prediction
    - Of those: 80% → [MASK], 10% → random token, 10% → unchanged

    Returns:
      masked_ids:   token ids with some replaced by [MASK] or random tokens
      labels:       -100 for unmasked (ignore in loss), actual id for masked
      mask_positions: which positions were selected for prediction
    """
    vocab_size = len(token2id)
    mask_id = token2id['[MASK]']
    special_ids = {token2id[s] for s in ['[PAD]', '[CLS]', '[SEP]']}

    masked_ids = token_ids.copy()
    labels = [-100] * len(token_ids)  # -100 = ignore in cross-entropy loss
    mask_positions = []

    for i, token_id in enumerate(token_ids):
        # Don't mask special tokens
        if token_id in special_ids:
            continue

        # 15% chance of being selected for prediction
        if random.random() < mask_prob:
            labels[i] = token_id  # The model needs to predict the original
            mask_positions.append(i)

            r = random.random()
            if r < 0.80:
                # 80%: replace with [MASK]
                masked_ids[i] = mask_id
            elif r < 0.90:
                # 10%: replace with random token (not special)
                random_id = random.randint(5, vocab_size - 1)  # Skip special tokens
                masked_ids[i] = random_id
            # else: 10% — leave unchanged (masked_ids[i] stays as token_id)

    return masked_ids, labels, mask_positions


# ═══════════════════════════════════════════════════════
# Build our mini corpus
# ═══════════════════════════════════════════════════════
corpus = [
    "The cat sat on the mat and licked its paws",
    "The dog ran across the park chasing the ball",
    "Paris is the capital city of France in Europe",
    "The weather in Paris is beautiful in springtime",
    "Machine translation helps people communicate across languages",
    "The encoder reads the full sentence before the decoder starts",
]

token2id, id2token = build_simple_vocab(corpus)

print("═" * 65)
print("  🎭 Masked Language Modeling: Fill-in-the-Blank Training")
print("═" * 65)
print(f"\n📚 Vocabulary size: {len(token2id)} tokens")
print(f"Special tokens: [PAD]=0, [UNK]=1, [MASK]=2, [CLS]=3, [SEP]=4")
print()

# Apply MLM to a few sentences
demo_sentences = corpus[:3]

for sentence in demo_sentences:
    tokens, token_ids = tokenize(sentence, token2id)
    masked_ids, labels, mask_positions = apply_mlm_masking(
        token_ids, token2id, id2token, mask_prob=0.15
    )

    print(f"Original: \"{sentence}\"")
    print(f"Tokens:   {tokens}")

    # Show the masked version
    masked_tokens = [id2token[i] for i in masked_ids]
    print(f"Masked:   {masked_tokens}")

    # Show what needs to be predicted
    if mask_positions:
        print(f"Predict:  ", end='')
        for pos in mask_positions:
            original_word = id2token[labels[pos]]
            what_model_sees = masked_tokens[pos]
            strategy = '→ [MASK]' if what_model_sees == '[MASK]' else (
                '→ random' if what_model_sees != original_word else '→ kept')
            print(f"pos {pos}: '{original_word}' {strategy}  ", end='')
        print()
    else:
        print(f"Predict:  (no tokens selected this round — try again!)")
    print()

# Show the masking statistics with a larger sample
print("─" * 65)
print("📊 Masking Strategy Statistics (over 1000 runs):")
mask_counts = {'[MASK]': 0, 'random': 0, 'unchanged': 0}
total_selected = 0
tokens_sample = [t for t in token2id if not t.startswith('[')]

for _ in range(1000):
    token_id = random.randint(5, len(token2id) - 1)
    r = random.random()
    if r < 0.80:
        mask_counts['[MASK]'] += 1
    elif r < 0.90:
        mask_counts['random'] += 1
    else:
        mask_counts['unchanged'] += 1
    total_selected += 1

for strategy, count in mask_counts.items():
    bar = '█' * int(count / 10)
    print(f"  {strategy:12}: {count:4d}/1000 ({count/10:.1f}%)  {bar}")

print("\n💡 KEY INSIGHT: By seeing both directions (left AND right context),")
print("   MLM trains a BIDIRECTIONAL representation — perfect for the encoder!")
print("   GPT-style next-token prediction only sees LEFT context — great for")
print("   generation but weaker for understanding.")

## 🎓 Supervised Fine-tuning: Learning to Translate

### From General Language Understanding to Translation

Pretraining gives the model a foundation — it understands language deeply. But it doesn't know how to translate yet. That's where supervised fine-tuning comes in.

**Bilingual vs Multilingual Approach:**

| Approach | Training Data | Model Count | Pros | Cons |
|---|---|---|---|---|
| **Bilingual** | EN-FR pairs, EN-DE pairs... | One per pair (N² models!) | Highest quality per pair | Doesn't scale to 100+ languages |
| **Multilingual** | All language pairs together | ONE model for all | Scales, cross-lingual transfer | Slightly lower quality, "curse of multilinguality" |

**Google's trick:** Add a language tag at the start!

```
MULTILINGUAL TRANSLATION WITH LANGUAGE TAGS:
═══════════════════════════════════════════

Input:  "<2fr> The cat sat on the mat"
Output: "Le chat s'est assis sur le tapis"

Input:  "<2de> The cat sat on the mat"
Output: "Die Katze saß auf der Matte"

Input:  "<2ja> The cat sat on the mat"
Output: "猫がマットの上に座っていた"

Same model, same weights! The <2XX> token tells the decoder which
language to generate. This means with N languages, you need:
  Bilingual: N × (N-1) / 2 models
  Multilingual: 1 model  ← Google uses this!
```

### The Training Objective

```
For each (source, target) pair in the parallel corpus:

  Encoder input:  [<2fr>] [The] [cat] [sat] [on] [the] [mat]
  Decoder input:  [<BOS>] [Le] [chat] [s'est] [assis] [sur] [le]  (teacher forcing)
  Decoder target: [Le]  [chat] [s'est] [assis]  [sur]  [le] [tapis] [<EOS>]

Loss = -∑ log P(target_t | source, target_<t)
       ↑ cross-entropy over all target tokens

"Teacher forcing": During training, we feed the ACTUAL previous target tokens
as decoder input (not the model's own predictions). This stabilizes training!
```


In [ ]:
"""
📝 Translation Training Data Format

Show how bilingual training pairs are formatted for encoder-decoder training:
  - Language tags for multilingual models
  - Teacher forcing: what encoder sees vs decoder input vs decoder target
  - How loss is computed token by token
"""

import numpy as np

# ═══════════════════════════════════════════════════════
# Parallel corpus: English-French sentence pairs
# ═══════════════════════════════════════════════════════
parallel_corpus = [
    ("The cat sat on the mat",        "Le chat s'est assis sur le tapis"),
    ("I love learning new languages",  "J'adore apprendre de nouvelles langues"),
    ("The weather is nice today",      "Il fait beau aujourd'hui"),
    ("Where is the nearest bakery?",   "Où est la boulangerie la plus proche?"),
    ("Machine learning is fascinating", "L'apprentissage automatique est fascinant"),
]


def format_training_pair(src, tgt, target_lang_code='fr'):
    """
    Format a source-target pair for encoder-decoder training.

    Returns a dict showing:
      encoder_input: source sentence with language tag
      decoder_input: target sentence shifted right (teacher forcing)
      decoder_target: target sentence shifted left (what to predict)
    """
    lang_tag = f'<2{target_lang_code}>'

    # Tokenize (word-level for display clarity)
    src_tokens = src.split()
    tgt_tokens = tgt.split()

    encoder_input = [lang_tag] + src_tokens
    decoder_input = ['<BOS>'] + tgt_tokens        # What decoder SEES (teacher forcing)
    decoder_target = tgt_tokens + ['<EOS>']       # What decoder needs to PREDICT

    assert len(decoder_input) == len(decoder_target), "Must be same length!"

    return {
        'encoder_input': encoder_input,
        'decoder_input': decoder_input,
        'decoder_target': decoder_target,
    }


print("═" * 70)
print("  📝 Translation Training Data Format (Teacher Forcing)")
print("═" * 70)

for i, (src, tgt) in enumerate(parallel_corpus[:3]):
    pair = format_training_pair(src, tgt)

    print(f"\n── Pair {i+1} ──────────────────────────────────────────────")
    print(f"🇬🇧 English:  '{src}'")
    print(f"🇫🇷 French:   '{tgt}'")
    print()
    print(f"ENCODER INPUT  (what the encoder reads):")
    print(f"  {pair['encoder_input']}")
    print(f"  ↑ Language tag tells the decoder: 'generate French!'")
    print()
    print(f"DECODER INPUT  (what the decoder sees — teacher forcing):")
    print(f"  {pair['decoder_input']}")
    print()
    print(f"DECODER TARGET (what the decoder must predict — labels):")
    print(f"  {pair['decoder_target']}")
    print()

    # Show the step-by-step prediction
    print("  Step-by-step prediction:")
    for step, (sees, predicts) in enumerate(zip(pair['decoder_input'], pair['decoder_target'])):
        print(f"    Step {step+1}: Decoder sees '{sees}' → must predict '{predicts}'")

print("\n" + "═" * 70)
print("  📊 Corpus Statistics")
print("═" * 70)
src_lens = [len(s.split()) for s, _ in parallel_corpus]
tgt_lens = [len(t.split()) for _, t in parallel_corpus]
print(f"  Pairs in corpus:        {len(parallel_corpus)}")
print(f"  Avg English length:     {np.mean(src_lens):.1f} tokens")
print(f"  Avg French length:      {np.mean(tgt_lens):.1f} tokens")
print(f"  Avg length ratio (FR/EN): {np.mean(tgt_lens)/np.mean(src_lens):.2f}x")
print()
print("💡 Note: French is typically ~15-20% longer than English.")
print("   This length ratio varies by language pair and affects")
print("   the BLEU brevity penalty calculation!")

print("\n" + "═" * 70)
print("  🌐 Multilingual: How Language Tags Scale")
print("═" * 70)
languages = ['fr', 'de', 'es', 'it', 'pt', 'nl', 'pl', 'ru', 'zh', 'ja']
print(f"  With {len(languages)} target languages:")
print(f"    Bilingual approach: {len(languages)} separate models")
print(f"    Multilingual approach: 1 model with {len(languages)} language tags")
print(f"  Tags: {[f'<2{l}>' for l in languages]}")
print()
n = 100
print(f"  With {n} languages (like Google's GNMT):")
print(f"    Bilingual: {n*(n-1)//2:,} model pairs 😱")
print(f"    Multilingual: 1 model 🎉")

## 📐 BLEU Score: Measuring Translation Quality

### The Problem: How Do You Grade a Translation?

```
Reference: "The cat is on the mat"
Candidate: "The cat sat on the mat"

Is this a good translation? Mostly yes! But how do we turn that into a NUMBER?
```

BLEU (Bilingual Evaluation Understudy) — introduced by IBM in 2002 — is the industry standard.

### Step 1: N-gram Precision (But With a Twist)

```
NAIVE UNIGRAM PRECISION:
  Reference: "The cat is on the mat"  (6 words)
  Candidate: "the the the the the the"  (6 words)

  Each "the" appears in the reference! Precision = 6/6 = 100%  ← WRONG!

CLIPPED PRECISION (BLEU's fix):
  Max count of "the" in reference = 2 (appears twice)
  Clipped count of "the" = min(6, 2) = 2
  Clipped precision = 2/6 = 33%  ← Much more reasonable!

Apply this to unigrams, bigrams, trigrams, 4-grams and combine.
```

### Step 2: Brevity Penalty

```
Without BP, you could cheat by outputting just one word:
  Reference: "The cat is on the mat"  (6 words)
  Candidate: "The"  → 1 word, but it IS in the reference → 100% precision!

Brevity Penalty (BP):
  r = reference length
  c = candidate length

  BP = 1            if c >= r  (no penalty if candidate is long enough)
  BP = exp(1 - r/c) if c < r   (exponential penalty for short candidates)

  c=1, r=6: BP = exp(1 - 6/1) = exp(-5) ≈ 0.007  ← massive penalty!
```

### Step 3: The Full BLEU Formula

```
BLEU = BP × exp( Σ wₙ × log(pₙ) )
       ↑         ↑    ↑    ↑
  brevity  weights (each  log-average of
  penalty  usually 1/N)   n-gram precisions

Standard: N=4, each wₙ = 0.25 (equal weights)

BLEU-4 = BP × exp( 0.25×log(p₁) + 0.25×log(p₂) + 0.25×log(p₃) + 0.25×log(p₄) )

Score range: 0 to 1 (or 0 to 100 as a percentage)
Human-level: ~0.40-0.60 on WMT benchmarks
State-of-art: ~0.45+ on WMT EN-DE
```

### BLEU's Known Weaknesses

| Weakness | Example | Impact |
|---|---|---|
| **No synonyms** | "big" ≠ "large" even if meaning is same | Underestimates quality |
| **No word order** | "cat the mat on sat" scores same as correct order | Misleading |
| **Precision only** | Ignores whether all reference content is covered | Miss long sentences |
| **No semantics** | Completely wrong sentence can have high BLEU | Fooled by tricks |


In [ ]:
"""
📐 BLEU Score from Scratch!

Implementing the full BLEU score step by step.
Test case: reference="The cat is on the mat", candidate="The cat sat on the mat"
"""

import math
from collections import Counter


def get_ngrams(tokens, n):
    """Extract all n-grams from a token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def clipped_ngram_precision(candidate_tokens, reference_tokens, n):
    """
    Compute clipped n-gram precision.

    Clipping: count each candidate n-gram at most as many times as
    it appears in the reference.

    Returns: (numerator, denominator, precision)
    """
    candidate_ngrams = get_ngrams(candidate_tokens, n)
    reference_ngrams = get_ngrams(reference_tokens, n)

    if not candidate_ngrams:
        return 0, 0, 0.0

    # Count n-grams in reference (the "ceiling" for clipping)
    ref_counts = Counter(reference_ngrams)

    # Count n-grams in candidate
    cand_counts = Counter(candidate_ngrams)

    # Clipped counts: min(candidate count, reference count)
    clipped_numerator = sum(
        min(count, ref_counts.get(ngram, 0))
        for ngram, count in cand_counts.items()
    )

    denominator = len(candidate_ngrams)
    precision = clipped_numerator / denominator if denominator > 0 else 0.0

    return clipped_numerator, denominator, precision


def brevity_penalty(candidate_len, reference_len):
    """
    Compute BLEU brevity penalty.
    BP = 1 if c >= r, else exp(1 - r/c)
    """
    if candidate_len >= reference_len:
        return 1.0
    else:
        return math.exp(1 - reference_len / candidate_len)


def bleu_score(candidate, reference, max_n=4, weights=None):
    """
    Compute BLEU score.

    Args:
        candidate: string (machine translation output)
        reference: string (human reference translation)
        max_n: maximum n-gram order (BLEU-4 uses 4)
        weights: list of weights for each n-gram order (uniform by default)

    Returns:
        dict with full breakdown of the calculation
    """
    if weights is None:
        weights = [1.0 / max_n] * max_n

    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()

    # Compute BP
    bp = brevity_penalty(len(cand_tokens), len(ref_tokens))

    # Compute n-gram precisions
    precisions = {}
    log_sum = 0.0

    for n in range(1, max_n + 1):
        numerator, denominator, precision = clipped_ngram_precision(
            cand_tokens, ref_tokens, n
        )
        precisions[n] = {
            'numerator': numerator,
            'denominator': denominator,
            'precision': precision,
        }

        if precision > 0:
            log_sum += weights[n-1] * math.log(precision)
        else:
            # If any precision is 0, BLEU is 0 (can't take log(0))
            return {
                'bleu': 0.0,
                'bp': bp,
                'precisions': precisions,
                'note': f'BLEU=0 because p_{n}=0'
            }

    bleu = bp * math.exp(log_sum)

    return {
        'bleu': bleu,
        'bp': bp,
        'precisions': precisions,
        'candidate_len': len(cand_tokens),
        'reference_len': len(ref_tokens),
    }


# ═══════════════════════════════════════════════════════
# Test cases
# ═══════════════════════════════════════════════════════
reference = "The cat is on the mat"

test_cases = [
    ("The cat is on the mat",       "Perfect match"),
    ("The cat sat on the mat",       "One word different (sat vs is)"),
    ("The cat is on the floor",      "One word different (floor vs mat)"),
    ("A dog sat on a rug",          "Completely wrong"),
    ("The the the the the the",     "Repetition attack"),
    ("The cat",                      "Too short (brevity penalty)"),
    ("The cat is on the mat today", "One extra word"),
]

print("═" * 70)
print("  📐 BLEU Score: Step-by-Step Calculation")
print("═" * 70)
print(f"  Reference: \"{reference}\"\n")

# Detailed breakdown for the main test case
candidate_main = "The cat sat on the mat"
result = bleu_score(candidate_main, reference)

print(f"── Detailed Breakdown for: \"{candidate_main}\" ──")
print(f"  Candidate length: {result['candidate_len']} | Reference length: {result['reference_len']}")
print(f"  Brevity Penalty: {result['bp']:.4f} (no penalty — same length)")
print()
for n, p_data in result['precisions'].items():
    print(f"  {n}-gram precision: {p_data['numerator']}/{p_data['denominator']} = {p_data['precision']:.4f}")
    # Show the actual n-grams
    cand_ngrams = get_ngrams(candidate_main.lower().split(), n)
    ref_ngrams_counts = Counter(get_ngrams(reference.lower().split(), n))
    matches = [ng for ng in cand_ngrams if ng in ref_ngrams_counts]
    no_matches = [ng for ng in cand_ngrams if ng not in ref_ngrams_counts]
    if matches:
        print(f"    ✅ Matches: {[' '.join(ng) for ng in set(matches)]}")
    if no_matches:
        print(f"    ❌ Misses:  {[' '.join(ng) for ng in set(no_matches)]}")

print(f"\n  BLEU = {result['bp']:.4f} × exp({sum(0.25*math.log(result['precisions'][n]['precision']) for n in range(1,5) if result['precisions'][n]['precision'] > 0):.4f})")
print(f"  BLEU = {result['bleu']:.4f}  ({result['bleu']*100:.2f}%)")

# Summary table for all test cases
print("\n" + "─" * 70)
print(f"  {'Candidate':<42} {'BLEU':>6}  {'BP':>6}  {'p1':>5}  {'p2':>5}")
print("─" * 70)

for candidate, description in test_cases:
    r = bleu_score(candidate, reference)
    bleu_val = r['bleu']
    bp_val = r['bp']
    p1 = r['precisions'][1]['precision']
    p2 = r['precisions'][2]['precision']
    print(f"  {candidate:<42} {bleu_val:>6.3f}  {bp_val:>6.3f}  {p1:>5.3f}  {p2:>5.3f}")
    print(f"  → {description}")

print("\n💡 Note the 'the the the' case: p1=0.333 (clipping works!), not 1.0")
print("   The brevity penalty kills the 'too short' candidate.")

## 🔴 ROUGE: Recall-Oriented Evaluation

ROUGE (Recall-Oriented Understudy for Gisting Evaluation) flips the question:

```
BLEU asks:  "Of the words I generated, how many are correct?"  → Precision
ROUGE asks: "Of the words in the reference, how many did I capture?" → Recall

Example:
  Reference:  "The cat is sitting on the mat"
  Candidate:  "The cat"

  BLEU:  2/2 = 100% unigram precision  ← BLEU likes this!
  ROUGE: 2/7 = 28.6% recall           ← ROUGE hates this!
```

### ROUGE Variants

| Variant | What It Measures | Formula |
|---|---|---|
| **ROUGE-1** | Single word overlap | Unigram recall |
| **ROUGE-2** | Two-word phrase overlap | Bigram recall |
| **ROUGE-L** | Longest common subsequence | Structural similarity |

### When to Use ROUGE vs BLEU

```
BLEU → best for translation (precision matters: every word counts)
ROUGE → best for summarization (recall matters: don't miss key facts)

BLEU punishes: wrong words in candidate
ROUGE punishes: missing words from reference
```

In practice, you want BOTH:
- High BLEU = the words you generated are good
- High ROUGE = you covered the reference content
- Their harmonic mean ≈ F1 score of translation quality


In [ ]:
"""
🔴 ROUGE-1 and ROUGE-2 from Scratch!

Implementing recall-oriented n-gram overlap.
"""

from collections import Counter


def rouge_n(candidate, reference, n):
    """
    Compute ROUGE-N score (recall, precision, F1).

    ROUGE-N Recall    = matching n-grams / total n-grams in REFERENCE
    ROUGE-N Precision = matching n-grams / total n-grams in CANDIDATE
    ROUGE-N F1        = 2 * (P * R) / (P + R)

    Returns dict with recall, precision, f1, and details.
    """
    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()

    cand_ngrams = Counter(get_ngrams(cand_tokens, n))
    ref_ngrams = Counter(get_ngrams(ref_tokens, n))

    # Count matching n-grams (clipped to min of candidate and reference counts)
    matches = sum(
        min(count, cand_ngrams.get(ngram, 0))
        for ngram, count in ref_ngrams.items()
    )

    total_ref = sum(ref_ngrams.values())
    total_cand = sum(cand_ngrams.values())

    recall    = matches / total_ref  if total_ref  > 0 else 0.0
    precision = matches / total_cand if total_cand > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)

    # Which n-grams matched?
    matched_ngrams = [
        ngram for ngram in ref_ngrams
        if ngram in cand_ngrams
    ]

    return {
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'matches': matches,
        'total_ref': total_ref,
        'total_cand': total_cand,
        'matched_ngrams': matched_ngrams,
    }


def rouge_l(candidate, reference):
    """
    ROUGE-L: Based on Longest Common Subsequence (LCS).
    LCS doesn't require consecutive matches — just same-order words.

    Uses dynamic programming.
    """
    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()

    m, n = len(ref_tokens), len(cand_tokens)

    # Build LCS table
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_tokens[i-1] == cand_tokens[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])

    lcs_length = dp[m][n]
    recall = lcs_length / m if m > 0 else 0.0
    precision = lcs_length / n if n > 0 else 0.0
    f1 = (2 * precision * recall / (precision + recall)
          if (precision + recall) > 0 else 0.0)

    return {'recall': recall, 'precision': precision, 'f1': f1, 'lcs_length': lcs_length}


# ═══════════════════════════════════════════════════════
# Test cases
# ═══════════════════════════════════════════════════════
reference = "The cat is on the mat"

test_cases = [
    ("The cat is on the mat",         "Perfect match"),
    ("The cat sat on the mat",         "One word different"),
    ("The cat",                        "Short — precision=1.0, recall=low"),
    ("The cat is on the mat period",   "One extra word — recall=1.0, precision<1.0"),
    ("mat the on is cat the",          "Correct words, wrong order"),
    ("A dog ran across the park",      "Wrong content"),
]

print("═" * 72)
print("  🔴 ROUGE Scores: Recall-Oriented Evaluation")
print("═" * 72)
print(f"  Reference: \"{reference}\"\n")

# Detailed breakdown for main case
main_cand = "The cat sat on the mat"
r1 = rouge_n(main_cand, reference, n=1)
r2 = rouge_n(main_cand, reference, n=2)
rl = rouge_l(main_cand, reference)

print(f"── Detailed Breakdown for: \"{main_cand}\" ──")
print()
print(f"  ROUGE-1 (unigrams):")
print(f"    Matched: {r1['matches']} n-grams  |  Ref total: {r1['total_ref']}  |  Cand total: {r1['total_cand']}")
print(f"    Matched n-grams: {[' '.join(ng) for ng in r1['matched_ngrams']]}")
print(f"    Recall={r1['recall']:.3f}, Precision={r1['precision']:.3f}, F1={r1['f1']:.3f}")
print()
print(f"  ROUGE-2 (bigrams):")
print(f"    Matched: {r2['matches']} bigrams  |  Ref total: {r2['total_ref']}  |  Cand total: {r2['total_cand']}")
print(f"    Matched bigrams: {[' '.join(ng) for ng in r2['matched_ngrams']]}")
print(f"    Recall={r2['recall']:.3f}, Precision={r2['precision']:.3f}, F1={r2['f1']:.3f}")
print()
print(f"  ROUGE-L (LCS):")
print(f"    LCS length={rl['lcs_length']}  |  Ref len={len(reference.split())}  |  Cand len={len(main_cand.split())}")
print(f"    Recall={rl['recall']:.3f}, Precision={rl['precision']:.3f}, F1={rl['f1']:.3f}")

# Summary table
print("\n" + "─" * 72)
header = f"  {'Candidate':<38}  {'R1-F1':>6}  {'R2-F1':>6}  {'RL-F1':>6}"
print(header)
print("─" * 72)

for candidate, description in test_cases:
    r1 = rouge_n(candidate, reference, n=1)
    r2 = rouge_n(candidate, reference, n=2)
    rl = rouge_l(candidate, reference)
    print(f"  {candidate:<38}  {r1['f1']:>6.3f}  {r2['f1']:>6.3f}  {rl['f1']:>6.3f}")
    print(f"  → {description}")

print()
print("💡 Notice: 'mat the on is cat the' (scrambled) has ROUGE-1 = 1.0!")
print("   All unigrams are present — ROUGE doesn't care about order!")
print("   That's why ROUGE-L (LCS) is better for order-sensitive tasks.")

## ⭐ METEOR: Synonym-Aware, Order-Sensitive Evaluation

METEOR (Metric for Evaluation of Translation with Explicit Ordering) was designed to fix BLEU's biggest flaws.

### What Makes METEOR Special?

```
BLEU/ROUGE problem:
  Reference: "The cat is sitting on the mat"
  Candidate: "The feline is sitting on the rug"

  BLEU sees: "feline" ≠ "cat", "rug" ≠ "mat" → low score
  Humans see: This is a GREAT translation! Synonyms are fine.

METEOR's fix: Match via stems AND synonyms!
  "feline" matches "cat" (synonym via WordNet)
  "rug" matches "mat" (synonym via WordNet)
  → Higher, more accurate score ✅
```

### METEOR's Three Matching Stages

```
STAGE 1: Exact word match
  "The" = "The" ✅  "cat" = "cat" ✅

STAGE 2: Stem match (e.g., Porter stemmer)
  "running" stem = "run" = stem of "runs" ✅
  "walked" stem = "walk" = stem of "walking" ✅

STAGE 3: Synonym match (via WordNet)
  "large" synset includes "big", "great", "huge" ✅
  "happy" synset includes "glad", "joyful", "pleased" ✅
```

### METEOR's Chunk Penalty (Order Matters!)

```
Even if all words match, METEOR penalizes scrambled order:

  Reference: "The cat is on the mat"
  Candidate A: "The cat is on the mat"   → 1 chunk (all in order) → no penalty
  Candidate B: "mat the on is cat The"   → 6 chunks (all scrambled) → big penalty

Chunk penalty = 0.5 × (chunks / matched_unigrams)^3
  → Exponential penalty for many chunks (scrambled text)
  → No penalty if everything is in order

Final METEOR = F-mean × (1 - chunk_penalty)
  where F-mean = 10PR / (R + 9P)  ← recall-weighted harmonic mean
```

### Quick Comparison: BLEU vs ROUGE vs METEOR

| | BLEU | ROUGE | METEOR |
|---|---|---|---|
| **Orientation** | Precision | Recall | Balanced (recall-weighted) |
| **Synonyms?** | ❌ | ❌ | ✅ |
| **Stemming?** | ❌ | ❌ | ✅ |
| **Word order?** | Partial (n-grams) | Partial | ✅ (chunk penalty) |
| **Speed** | Fast ⚡ | Fast ⚡ | Slower 🐌 |
| **Best for** | Translation | Summarization | Translation (more nuanced) |
| **Correlation with humans** | Good | Good | Better |


In [ ]:
"""
⭐ METEOR-like Calculation from Scratch

A simplified METEOR implementation that demonstrates:
  1. Exact + stem matching (no full WordNet, but shows the concept)
  2. F-mean (recall-weighted harmonic mean)
  3. Chunk penalty for word order

In production, you'd use the full METEOR toolkit with WordNet synonyms.
This shows the CONCEPT clearly.
"""

import re
from collections import defaultdict


def simple_stem(word):
    """
    A very simple stemmer (subset of Porter stemmer rules).
    Real METEOR uses a proper stemmer like NLTK's PorterStemmer.
    """
    word = word.lower()
    # Remove common suffixes
    for suffix in ['ing', 'tion', 'ness', 'ed', 'er', 'est', 'ly', 's']:
        if word.endswith(suffix) and len(word) > len(suffix) + 2:
            return word[:-len(suffix)]
    return word


# Simple synonym dictionary (subset of WordNet for demo)
SYNONYM_GROUPS = [
    {'cat', 'feline', 'kitty', 'kitten'},
    {'dog', 'canine', 'hound', 'puppy'},
    {'mat', 'rug', 'carpet', 'floor'},
    {'big', 'large', 'huge', 'great', 'enormous'},
    {'happy', 'joyful', 'glad', 'pleased', 'delighted'},
    {'run', 'sprint', 'dash', 'race', 'jog'},
    {'sit', 'rest', 'perch', 'settle'},
    {'quick', 'fast', 'rapid', 'speedy', 'swift'},
]

def get_synonym_group(word):
    """Return the synonym group for a word (or just {word} if not found)."""
    word = word.lower()
    for group in SYNONYM_GROUPS:
        if word in group:
            return group
        # Also check stems
        if simple_stem(word) in {simple_stem(w) for w in group}:
            return group
    return {word}


def meteor_match(cand_tokens, ref_tokens):
    """
    Find best alignment between candidate and reference tokens.
    Match via:
      1. Exact match
      2. Stem match
      3. Synonym match

    Returns: list of (cand_idx, ref_idx) matched pairs
    """
    matches = []  # (cand_idx, ref_idx)
    used_ref = set()
    used_cand = set()

    # Pass 1: Exact matches
    for ci, cw in enumerate(cand_tokens):
        for ri, rw in enumerate(ref_tokens):
            if ri not in used_ref and ci not in used_cand:
                if cw.lower() == rw.lower():
                    matches.append((ci, ri))
                    used_ref.add(ri)
                    used_cand.add(ci)
                    break

    # Pass 2: Stem matches
    for ci, cw in enumerate(cand_tokens):
        if ci in used_cand:
            continue
        for ri, rw in enumerate(ref_tokens):
            if ri not in used_ref:
                if simple_stem(cw) == simple_stem(rw):
                    matches.append((ci, ri))
                    used_ref.add(ri)
                    used_cand.add(ci)
                    break

    # Pass 3: Synonym matches
    for ci, cw in enumerate(cand_tokens):
        if ci in used_cand:
            continue
        cand_syns = get_synonym_group(cw)
        for ri, rw in enumerate(ref_tokens):
            if ri not in used_ref:
                ref_syns = get_synonym_group(rw)
                if cand_syns & ref_syns:  # Non-empty intersection
                    matches.append((ci, ri))
                    used_ref.add(ri)
                    used_cand.add(ci)
                    break

    return sorted(matches, key=lambda x: x[0])  # Sort by candidate position


def count_chunks(matches):
    """
    Count the number of "chunks" in the alignment.
    A chunk = a consecutive sequence of matched tokens in the same order.
    More chunks = more scrambled order = higher penalty.
    """
    if not matches:
        return 0
    chunks = 1
    for i in range(1, len(matches)):
        # A new chunk starts when the relative order breaks
        ci_prev, ri_prev = matches[i-1]
        ci_curr, ri_curr = matches[i]
        if ci_curr != ci_prev + 1 or ri_curr != ri_prev + 1:
            chunks += 1
    return chunks


def meteor_score(candidate, reference, alpha=0.9, beta=3.0, gamma=0.5):
    """
    Compute METEOR score.

    Args:
        alpha: Weight for recall in F-mean (0.9 means recall is 9x more important)
        beta:  Chunk penalty exponent (default 3.0)
        gamma: Chunk penalty coefficient (default 0.5)

    Returns: dict with METEOR score and breakdown
    """
    cand_tokens = candidate.lower().split()
    ref_tokens = reference.lower().split()

    # Find best alignment
    matches = meteor_match(cand_tokens, ref_tokens)
    m = len(matches)

    if m == 0:
        return {'meteor': 0.0, 'precision': 0.0, 'recall': 0.0,
                'f_mean': 0.0, 'chunk_penalty': 1.0, 'matches': 0}

    precision = m / len(cand_tokens)
    recall    = m / len(ref_tokens)

    # Recall-weighted F-mean (recall gets 9x more weight when alpha=0.9)
    f_mean = precision * recall / (alpha * recall + (1 - alpha) * precision)

    # Chunk penalty: penalize scrambled order
    num_chunks = count_chunks(matches)
    chunk_penalty = gamma * (num_chunks / m) ** beta

    meteor = f_mean * (1 - chunk_penalty)

    # Identify match types for display
    match_details = []
    for ci, ri in matches:
        cw, rw = cand_tokens[ci], ref_tokens[ri]
        if cw.lower() == rw.lower():
            match_type = 'exact'
        elif simple_stem(cw) == simple_stem(rw):
            match_type = 'stem'
        else:
            match_type = 'synonym'
        match_details.append((cw, rw, match_type))

    return {
        'meteor': meteor,
        'precision': precision,
        'recall': recall,
        'f_mean': f_mean,
        'chunk_penalty': chunk_penalty,
        'num_chunks': num_chunks,
        'matches': m,
        'match_details': match_details,
    }


# ═══════════════════════════════════════════════════════
# Test cases designed to show METEOR's advantages
# ═══════════════════════════════════════════════════════
reference = "The cat is on the mat"

test_cases = [
    ("The cat is on the mat",      "Perfect match"),
    ("The cat sat on the mat",      "One word different"),
    ("The feline is on the rug",    "SYNONYMS: feline=cat, rug=mat (BLEU misses!)"),
    ("The cat is sitting on mat",   "Stem: sitting≈sit (close)"),
    ("mat the on is cat The",       "Same words, scrambled order (chunk penalty!)"),
    ("A dog ran across the park",   "Wrong content"),
]

print("═" * 72)
print("  ⭐ METEOR Score: Synonym-Aware, Order-Sensitive")
print("═" * 72)
print(f"  Reference: \"{reference}\"\n")

# Detailed breakdown for synonym case (the interesting one!)
synonym_cand = "The feline is on the rug"
r = meteor_score(synonym_cand, reference)

print(f"── Detailed Breakdown for: \"{synonym_cand}\" ──")
print(f"  Total matches: {r['matches']} / {len(reference.split())} reference words")
print(f"  Match details:")
for cw, rw, mtype in r['match_details']:
    icon = '🎯' if mtype == 'exact' else ('🌱' if mtype == 'stem' else '🔀')
    print(f"    {icon} '{cw}' ↔ '{rw}' ({mtype} match)")
print(f"  Precision:      {r['precision']:.3f}")
print(f"  Recall:         {r['recall']:.3f}")
print(f"  F-mean:         {r['f_mean']:.3f}  (recall-weighted)")
print(f"  Chunks:         {r['num_chunks']} (fewer = more in-order)")
print(f"  Chunk penalty:  {r['chunk_penalty']:.4f}")
print(f"  METEOR:         {r['meteor']:.3f}")

print("\n" + "─" * 72)
print(f"  {'Candidate':<38}  {'METEOR':>7}  {'Note':<22}")
print("─" * 72)

for candidate, description in test_cases:
    r = meteor_score(candidate, reference)
    # Also compute BLEU-1 for comparison
    bleu_r = bleu_score(candidate, reference, max_n=1)
    print(f"  {candidate:<38}  {r['meteor']:>7.3f}  BLEU-1={bleu_r['bleu']:.3f}")
    print(f"  → {description}")

print()
print("💡 KEY INSIGHT: 'The feline is on the rug'")
print("   BLEU:   Low score (feline≠cat, rug≠mat by exact match)")
print("   METEOR: High score (recognizes feline=cat, rug=mat via synonyms)")
print("   This is why METEOR correlates better with human judgment!")

## 🏗️ System Design: Google Translate End-to-End

### The Full Production Pipeline

```
USER INPUT: "Translate 'Where is the library?' to French"
                              │
                              ▼
          ┌─────────────────────────────────────┐
          │  1. LANGUAGE DETECTOR               │
          │     Input: raw text                 │
          │     Output: detected language code  │
          │     "en" (English detected!)        │
          │     Model: character n-gram         │
          │     classifier (fastText/CLD)       │
          └──────────────────┬──────────────────┘
                             │
                             ▼
          ┌─────────────────────────────────────┐
          │  2. PRE-PROCESSING                  │
          │     - Script detection              │
          │     - Normalization (unicode)       │
          │     - Segmentation (sentences)      │
          └──────────────────┬──────────────────┘
                             │
                             ▼
          ┌─────────────────────────────────────┐
          │  3. BPE TOKENIZER                   │
          │     Input: normalized text          │
          │     Output: subword token IDs       │
          │     Prepend: <2fr> language tag     │
          └──────────────────┬──────────────────┘
                             │
                             ▼
          ┌─────────────────────────────────────┐
          │  4. ENCODER-DECODER TRANSFORMER     │
          │     - Encoder: bidirectional        │
          │     - Decoder: autoregressive       │
          │     - Beam search (k=4) for output  │
          │     Model: mBART / NLLB-200         │
          └──────────────────┬──────────────────┘
                             │
                             ▼
          ┌─────────────────────────────────────┐
          │  5. DETOKENIZER + POST-PROCESSING   │
          │     - BPE → words                  │
          │     - Truecasing                   │
          │     - Punctuation normalization     │
          └──────────────────┬──────────────────┘
                             │
                             ▼
         OUTPUT: "Où est la bibliothèque?"
```

### Production Considerations

| Challenge | Solution |
|---|---|
| **Latency < 200ms** | INT8 quantization, model distillation, GPU batching, KV cache |
| **100+ languages** | Single multilingual model (NLLB-200), shared BPE vocab |
| **Low-resource languages** | Backtranslation augmentation, pivot via high-resource language |
| **Quality monitoring** | Sample 0.1% for human eval, BLEU on held-out test sets, drift alerts |
| **Safety** | Content filtering pre/post translation, bias audits per language pair |
| **Cost** | Smaller student model via distillation, cache frequent phrases, batch idle requests |

### Beam Search: Choosing the Best Translation

```
Greedy decoding: always pick the HIGHEST probability next token
  → Fast but suboptimal (may miss better translations)

Beam search (k=4): keep the k BEST partial translations at each step
  → Explores multiple paths, finds better overall sequences

Step 1: "Le" (p=0.6), "La" (p=0.3), "Un" (p=0.07), "Ce" (p=0.03)
        Keep top 4 beams ───────────────────────────────────────────►
Step 2: "Le chat" (0.6×0.8=0.48), "Le chien" (0.6×0.1=0.06),
        "La chatte" (0.3×0.7=0.21), ...
        Keep top 4 beams ───────────────────────────────────────────►
...
Final: Pick the sequence with highest total log-probability (divided by length)
```


In [ ]:
"""
🌐 Language Detector: Character N-gram Classifier

Language detection is the FIRST step in the Google Translate pipeline.
We'll implement a simple character n-gram based language identifier.

Real production systems (CLD3, fastText) use larger feature sets,
but the core concept is the same: character n-gram profiles per language.
"""

import re
import math
from collections import Counter, defaultdict


def get_char_ngrams(text, n=3, top_k=300):
    """
    Extract character n-grams from text.
    Return the top_k most frequent ones.
    """
    # Normalize: lowercase, keep only letters/spaces
    text = re.sub(r'[^a-zA-Záéíóúàèìòùäëïöüâêîôûçñ ]', '', text.lower())
    ngrams = Counter()
    for i in range(len(text) - n + 1):
        ngrams[text[i:i+n]] += 1
    return dict(ngrams.most_common(top_k))


def cosine_similarity(profile1, profile2):
    """
    Compute cosine similarity between two n-gram frequency profiles.
    Treats each profile as a vector in n-gram space.
    """
    all_keys = set(profile1.keys()) | set(profile2.keys())
    dot = sum(profile1.get(k, 0) * profile2.get(k, 0) for k in all_keys)
    mag1 = math.sqrt(sum(v**2 for v in profile1.values()))
    mag2 = math.sqrt(sum(v**2 for v in profile2.values()))
    if mag1 == 0 or mag2 == 0:
        return 0.0
    return dot / (mag1 * mag2)


class SimpleLanguageDetector:
    """Language detector using character n-gram cosine similarity."""

    def __init__(self, n=3):
        self.n = n
        self.profiles = {}  # lang_code → n-gram profile

    def fit(self, language_corpus):
        """
        Build language profiles from training text.
        language_corpus: dict {lang_code: text_string}
        """
        for lang_code, text in language_corpus.items():
            self.profiles[lang_code] = get_char_ngrams(text, self.n)
        return self

    def detect(self, text, top_n=3):
        """
        Detect language by comparing text's n-gram profile to all language profiles.
        Returns ranked list of (language, similarity_score).
        """
        text_profile = get_char_ngrams(text, self.n)
        scores = {
            lang: cosine_similarity(text_profile, profile)
            for lang, profile in self.profiles.items()
        }
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_n]


# ═══════════════════════════════════════════════════════
# Build language profiles
# ═══════════════════════════════════════════════════════
TRAINING_CORPUS = {
    'en': """
    The quick brown fox jumps over the lazy dog. Language detection is an
    important first step in machine translation systems. English text has
    distinctive patterns in its character sequences. The encoder reads the
    entire source sentence before the decoder generates the translation.
    Natural language processing enables computers to understand human language.
    Machine learning models are trained on large datasets of text examples.
    The transformer architecture revolutionized natural language processing.
    """,
    'fr': """
    Le renard brun rapide saute par-dessus le chien paresseux. La détection
    de la langue est une première étape importante dans les systèmes de
    traduction automatique. Le texte français a des schémas distinctifs dans
    ses séquences de caractères. Le traitement du langage naturel permet aux
    ordinateurs de comprendre le langage humain. Les modèles d'apprentissage
    automatique sont entraînés sur de grands ensembles de données textuelles.
    L'architecture du transformateur a révolutionné le traitement du langage.
    """,
    'de': """
    Der schnelle braune Fuchs springt über den faulen Hund. Die Spracherkennung
    ist ein wichtiger erster Schritt in maschinellen Übersetzungssystemen.
    Deutschsprachiger Text hat charakteristische Muster in seinen Zeichenfolgen.
    Die Transformer-Architektur hat die Verarbeitung natürlicher Sprache
    revolutioniert. Maschinelles Lernen ermöglicht es Computern, menschliche
    Sprache zu verstehen und zu verarbeiten.
    """,
    'es': """
    El rápido zorro marrón salta sobre el perro perezoso. La detección del idioma
    es un primer paso importante en los sistemas de traducción automática. El texto
    en español tiene patrones distintivos en sus secuencias de caracteres. El
    procesamiento del lenguaje natural permite a las computadoras entender el
    lenguaje humano. Los modelos de aprendizaje automático se entrenan en grandes
    conjuntos de datos de texto.
    """,
    'it': """
    La rapida volpe marrone salta sul cane pigro. Il rilevamento della lingua è
    un primo passo importante nei sistemi di traduzione automatica. Il testo
    italiano ha schemi caratteristici nelle sue sequenze di caratteri. Il
    modello di linguaggio trasformatore ha rivoluzionato l'elaborazione del
    linguaggio naturale. I modelli di apprendimento automatico vengono addestrati
    su grandi dataset di testo.
    """,
}

# Train the detector
detector = SimpleLanguageDetector(n=3)
detector.fit(TRAINING_CORPUS)

print("═" * 65)
print("  🌐 Language Detector: Character N-gram Classifier")
print("═" * 65)
print(f"\nTrained on {len(TRAINING_CORPUS)} languages: {list(TRAINING_CORPUS.keys())}")

# Print distinctive n-gram profiles
print("\n📊 Sample distinctive 3-grams per language:")
for lang in ['en', 'fr', 'de', 'es']:
    top_ngrams = sorted(detector.profiles[lang].items(),
                        key=lambda x: x[1], reverse=True)[:8]
    print(f"  {lang}: {[ng for ng, _ in top_ngrams]}")

# Test detection on new sentences
test_sentences = [
    ("Where is the nearest library?",          'en', "English"),
    ("Où est la bibliothèque la plus proche?",  'fr', "French"),
    ("Wo ist die nächste Bibliothek?",          'de', "German"),
    ("¿Dónde está la biblioteca más cercana?",  'es', "Spanish"),
    ("I love machine learning and deep learning", 'en', "English (technical)"),
    ("Ciao come stai oggi?",                    'it', "Italian"),
]

print("\n" + "═" * 65)
print("  🔍 Language Detection Results")
print("═" * 65)

correct = 0
for text, expected, description in test_sentences:
    predictions = detector.detect(text, top_n=3)
    top_lang, top_score = predictions[0]
    is_correct = top_lang == expected
    correct += int(is_correct)
    status = '✅' if is_correct else '❌'

    print(f"\n{status} '{text[:50]}...' " if len(text) > 50 else f"\n{status} '{text}'")
    print(f"   Expected: {expected} | Predicted: {top_lang} (score={top_score:.4f})")
    print(f"   Rankings: ", end='')
    for lang, score in predictions:
        print(f"{lang}={score:.3f}  ", end='')
    print()

print(f"\n📊 Accuracy: {correct}/{len(test_sentences)} = {correct/len(test_sentences)*100:.1f}%")

print("\n💡 Production note: Google uses CLD3 (Compact Language Detector v3)")
print("   which uses a neural network with 150+ character/word n-gram features.")
print("   It detects 107 languages with >99% accuracy on sentences >20 chars.")

## 🎯 Interview Walkthrough Cheat Sheet

### The 7-Step Framework Applied to Google Translate

| Step | What to Say | Key Terms to Drop |
|---|---|---|
| **1. Requirements** | "What language pairs? Real-time or batch? Latency target? Safety requirements?" | SLA, throughput, latency percentiles |
| **2. ML Framing** | "Sequence-to-sequence task. Input: source text (tokenized). Output: target text (autoregressive generation). Loss: cross-entropy over target vocabulary." | seq2seq, autoregressive, teacher forcing |
| **3. Data** | "Parallel corpora (e.g., Common Crawl, WMT). BPE tokenization (shared multilingual vocab). Backtranslation for low-resource augmentation." | parallel corpus, BPE, backtranslation |
| **4. Model** | "Encoder-decoder Transformer. Encoder: bidirectional self-attention. Decoder: causal self-attention + cross-attention. Multilingual with language tags." | encoder-decoder, cross-attention, NLLB |
| **5. Evaluation** | "Offline: BLEU (precision), ROUGE (recall), METEOR (synonyms + order). Online: user edit distance, engagement, A/B test." | BLEU, METEOR, human eval, A/B |
| **6. System Design** | "Language detector → BPE tokenizer → encoder-decoder → detokenizer → post-processing." | CLD3, beam search, KV cache |
| **7. Deployment** | "INT8 quantization + distillation for latency. Monitor BLEU drift, toxicity rates. Canary deploy, rollback trigger." | quantization, distillation, monitoring |

### Power Phrases That Impress Interviewers

```
ON ARCHITECTURE:
"We need an encoder-decoder rather than decoder-only because translation requires
full bidirectional understanding of the source before generating the target.
Word order can differ completely between languages — French 'ne...pas' splits the
verb, German puts the verb at the end in subordinate clauses."

ON CROSS-ATTENTION:
"Cross-attention is the key mechanism — the decoder's query vector at each step
attends to all encoder output vectors, effectively asking: which source words
are most relevant to generating this target word right now?"

ON BPE:
"BPE tokenization solves three problems at once: vocabulary explosion across
languages, OOV words, and morphological variation. With 50K shared subword
tokens, we cover 100+ languages without per-language vocabularies."

ON EVALUATION:
"BLEU is precision-oriented and treats synonyms as wrong. We supplement with
METEOR for synonym-aware evaluation and track human evaluation scores monthly.
For production, user edit distance (how much humans correct our output) is the
north star metric."

ON SCALING:
"Going multilingual creates a positive feedback loop: high-resource languages
like English improve low-resource ones like Swahili through shared representations.
NLLB-200 (Meta, 2022) showed 44% improvement on low-resource pairs by scaling
to 54B parameters with 200 languages."
```

### Common Follow-Up Questions & Strong Answers

| Question | Answer |
|---|---|
| "How do you handle very long documents?" | Sentence segmentation first (split at sentence boundaries), translate each sentence, then optional document-level post-processing for coreference consistency |
| "What about domain-specific translation (medical, legal)?" | Fine-tune on domain-specific parallel data. Use terminology constraints during beam search — force certain technical terms to appear in the output |
| "How do you handle low-resource languages?" | Three strategies: (1) multilingual transfer from related languages, (2) backtranslation to create synthetic pairs, (3) pivot translation through a high-resource intermediary language |
| "BLEU has known flaws — what do you use in production?" | Ensemble: BLEU for quick checks, METEOR for nuanced quality, human evaluation sampling (1% of translations), user edit rate as the ultimate ground truth |
| "How do you ensure the translation preserves tone/formality?" | This is an open research problem! Approaches: fine-tune on formal/informal pairs with style labels, train a formality classifier to guide beam search, use RLHF from human preference data |


## 🧪 Quiz Time! Training, Metrics & System Design

Final boss quiz — these are actual interview questions! Try before peeking. 🔒

---

### Question 1: BLEU's Clipping Mechanism

Given:
- Reference: `"the cat sat on the mat"`
- Candidate: `"the the the the the the"`

What is the clipped unigram precision?

- A) 6/6 = 100% (every word appears in the reference)
- B) 1/6 = 16.7% ("the" counts as just 1 unique type)
- C) 2/6 = 33.3% ("the" appears twice in reference, so max count = 2)
- D) 0/6 = 0% (repetition is penalized to zero)

<details>
<summary>🔓 Reveal Answer</summary>

**C) 2/6 = 33.3%** — This is exactly what BLEU's clipping is designed for. The candidate has 6 instances of "the", but the reference only has 2 instances of "the" ("the cat sat on **the** mat" — "the" appears at positions 1 and 5). Clipped count = min(6, 2) = 2. Clipped precision = 2/6 ≈ 0.333. Without clipping, naive precision would be 6/6 = 100%, which would be absurd.

</details>

---

### Question 2: MLM vs Next-Token Prediction

You're pretraining the ENCODER part of a translation model. Should you use Masked Language Modeling (MLM) or next-token prediction? Why?

- A) Next-token prediction — it's more efficient computationally
- B) MLM — because the encoder needs to see ALL surrounding context (bidirectional) to build rich representations of the full source sentence
- C) Either works equally well for encoders
- D) MLM — because it's the only method that works with multiple languages

<details>
<summary>🔓 Reveal Answer</summary>

**B) MLM!** The encoder must process the ENTIRE source sentence bidirectionally — each token's representation should incorporate information from both left and right context. MLM (like BERT) achieves this by training on predicting masked tokens using full bidirectional attention. Next-token prediction (like GPT) only uses left context, producing unidirectional representations — much weaker for the encoder. mBART, a leading multilingual translation model, uses a denoising/MLM-style pretraining specifically to get bidirectional encoders.

</details>

---

### Question 3: BLEU vs METEOR

You have two translation candidates:
- Reference: `"The large feline rested on the floor covering"`
- Candidate A: `"The large feline rested on the floor covering"` (perfect)
- Candidate B: `"The big cat lay on the rug"` (all synonyms, different order slightly)

Which metric would give Candidate B the HIGHEST relative score?

- A) BLEU-4 — because it measures all 4-gram overlaps
- B) ROUGE-2 — because it measures bigram recall
- C) METEOR — because it handles synonym matching (big=large, cat=feline, rug=floor covering, lay=rested)
- D) All metrics would give the same score

<details>
<summary>🔓 Reveal Answer</summary>

**C) METEOR!** Candidate B uses synonyms throughout: big≈large, cat≈feline, lay≈rested, rug≈floor covering. BLEU and ROUGE are exact/clipped match based — they would score Candidate B much lower than A because none of these synonym pairs match exactly. METEOR's synonym and stem matching (via WordNet) would recognize these equivalences and give Candidate B a much higher score, accurately reflecting that it's actually a high-quality translation from a human's perspective.

</details>

---

### Question 4: Backtranslation

You're building a translator for a low-resource language pair (English ↔ Swahili) with only 100,000 parallel sentence pairs. What is backtranslation and how does it help?

- A) Translating the model's output back to English to check for errors
- B) Using a Swahili→English model to translate large amounts of monolingual Swahili text into English, creating synthetic parallel pairs that augment your training data
- C) A data cleaning technique to remove bad translations from the corpus
- D) Fine-tuning the model in reverse (Swahili→English) to improve English→Swahili quality

<details>
<summary>🔓 Reveal Answer</summary>

**B)** Backtranslation (Sennrich et al., 2016) is a key data augmentation technique. If you have a Swahili→English model (even a weak one), use it to translate a large MONOLINGUAL Swahili corpus into English. This creates (synthetic_english, real_swahili) pairs — the Swahili side is authentic, only the English is machine-generated. These synthetic pairs dramatically increase training data for low-resource directions. It works because even imperfect backtranslations expose the model to more diverse, authentic target-side text. Papers show this can double BLEU scores for low-resource pairs!

</details>

---

### Question 5: System Bottleneck

Google Translate is serving 1 billion translation requests per day with p95 latency of 500ms — too slow. Your manager asks you to cut latency in half. What do you propose? (Pick the MOST impactful single intervention.)

- A) Use beam search width k=1 (greedy decoding) instead of k=4
- B) Apply INT8 quantization to reduce the model from FP32 to INT8 weights
- C) Train a smaller "student" model via knowledge distillation that mimics the large model but with 10x fewer parameters
- D) Cache the top 10,000 most frequently requested translations

<details>
<summary>🔓 Reveal Answer</summary>

**C) Knowledge distillation** is the highest-impact single change for inference latency at production scale. A 10x smaller student model can be 5-10x faster at inference while maintaining ~95% of quality (because the student learns from the teacher's soft output distributions, not just hard labels). INT8 quantization (B) gives ~2x speedup with minimal quality loss — good but smaller gain. Greedy decoding (A) helps ~4x since beam search runs k forward passes, but hurts quality significantly. Caching (D) only helps for repeated queries (rare for natural language). In practice, Google uses ALL of these together, but distillation gives the biggest quality-preserving speedup.

</details>
